# Structural breaks in Turkish macro-financial time series — analysis notebook

Runs the `structural_break_analyzer` ensemble on the series prepared by `01_data.py` and produces every table and figure of the paper.

Steps: (1) main panel, four monthly series, v = 5, with per-detector runs to recover cluster spans and vote composition · (2) daily USD/TRY, two windows, log level and 21-day realised volatility · (3) matching to the Table 1 chronology · (4) LLM attributions and their ratings against the record, for the main model and six further models (generated through OpenRouter if `OPENROUTER_API_KEY` is set and `NO_LLM` is not `1`; otherwise read from `output/attributions/`) · (5) tables and (6) figures → `output/` · (7) summary.

In [1]:
import os, io, re, json, contextlib, warnings, logging, datetime as dt
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore"); logging.disable(logging.CRITICAL)
for n in ("cmdstanpy","prophet","matplotlib"): logging.getLogger(n).setLevel(logging.CRITICAL)

import structural_break_analyzer as sba_pkg
from structural_break_analyzer import StructuralBreakAnalyzer
from structural_break_analyzer.detection.ensemble.voting_ensemble import VotingEnsembleDetector

HERE = os.getcwd(); DATA = os.path.join(HERE, "data", "processed"); OUT = os.path.join(HERE, "output")
for d in ("tables", "figures", "attributions"): os.makedirs(os.path.join(OUT, d), exist_ok=True)
CACHE_DIR = os.path.normpath(os.path.join(HERE, "..", "..", "turkey_structural_breaks", "results"))   # cached LLM run of 27 May 2026
MIN_VOTES = 5
METHODS = VotingEnsembleDetector().all_methods
SHORT = {"bai_perron":"Bai–Perron","cusum":"CUSUM","chow_test":"Chow","zivot_andrews":"Zivot–Andrews","pelt":"PELT",
         "binary_segmentation":"Binary seg.","dynamic_programming":"Dynamic prog.","mosum":"MOSUM","wild_binary_segmentation":"Wild binary seg.","prophet":"Prophet"}
SERIES = {  # key: (analysis column, label, unit for Δ, panel letter)
    "cpi_yoy":     ("level",     "CPI inflation (year-on-year, %)",   "pp",  "a"),
    "policy_rate": ("level",     "Overnight interbank rate (%)",      "pp",  "b"),
    "cci":         ("level",     "Consumer confidence index",         "pts", "c"),
    "tur_etf":     ("log_level", "MSCI Turkey ETF (TUR, USD)",        "%",   "d"),
}
def load(key): return pd.read_csv(os.path.join(DATA, f"{key}.csv"), parse_dates=["Date"])
def quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()): return fn(*a, **k)
import pickle, hashlib
from types import SimpleNamespace
CACHE = os.path.join(OUT, "cache"); os.makedirs(CACHE, exist_ok=True)
def _key(df, col, tag):
    h = hashlib.md5(pd.util.hash_pandas_object(df[["Date", col]], index=False).values.tobytes()).hexdigest()[:10]
    return os.path.join(CACHE, f"{tag}_{col}_{h}.pkl")
def _light(res):
    return SimpleNamespace(breaks=[SimpleNamespace(index=int(b.index), date=pd.Timestamp(b.date), confidence=float(b.confidence), magnitude=float(b.magnitude),
                                                   direction=b.direction, statistics=dict(b.statistics or {})) for b in res.breaks], value_col=getattr(res, "value_col", col_holder[0]))
col_holder = [None]
def detect(df, col, min_votes=MIN_VOTES):
    """ensemble run, cached on the data content"""
    f = _key(df, col, f"ensemble_v{min_votes}"); col_holder[0] = col
    if os.path.exists(f): return pickle.load(open(f, "rb"))
    res = _light(quiet(StructuralBreakAnalyzer().detect, df, value_col=col, date_col="Date", method="ensemble", min_votes=min_votes)); pickle.dump(res, open(f, "wb")); return res
def detect_single(df, col, method):
    f = _key(df, col, f"single_{method}"); col_holder[0] = col
    if os.path.exists(f): return pickle.load(open(f, "rb"))
    res = _light(quiet(StructuralBreakAnalyzer().detect, df, value_col=col, date_col="Date", method=method)); pickle.dump(res, open(f, "wb")); return res
print("structural_break_analyzer", getattr(sba_pkg, "__version__", "?"), "| detectors:", len(METHODS)); print(pd.read_csv(os.path.join(DATA, "_manifest.csv")).to_string(index=False))

structural_break_analyzer 2.0.0 | detectors: 10
     series                           label transform   n      start        end  interpolated
    tur_etf      MSCI Turkey ETF (TUR), USD log level 218 2008-04-01 2026-05-01             0
    cpi_yoy CPI inflation, year-on-year (%)     level 264 2004-01-01 2025-12-01             0
policy_rate    Overnight interbank rate (%)     level 243 2006-01-01 2026-03-01             9
        cci       Consumer confidence index     level 256 2005-01-01 2026-04-01             0


## 1. Main panel and per-detector replay

In [2]:
def tolerance(T): return 5 if T >= 200 else (max(3, min(5, T//40)) if T >= 50 else max(2, min(5, T//20)))
def replay_clusters(cands, tol):
    """package rule: sort by position, chain while gap to the cluster's most recent member <= tol"""
    cands = sorted(cands, key=lambda c: c[0]); clusters = []
    for c in cands:
        if clusters and c[0] - clusters[-1][-1][0] <= tol: clusters[-1].append(c)
        else: clusters.append([c])
    return clusters
def analyse(key, col, label, unit, min_votes=MIN_VOTES, with_replay=True):
    df = load(key); T = len(df); res = detect(df, col, min_votes); y = df[col].values
    sd_step = np.nanstd(np.diff(y))
    cands, per_method = [], {}
    if with_replay:
        for m in METHODS:
            try: r = detect_single(df, col, m); pts = [(b.index, b.confidence) for b in r.breaks]
            except Exception: pts = []
            per_method[m] = pts; cands += [(i, c, m) for i, c in pts]
        clusters = replay_clusters(cands, tolerance(T))
    rows = []
    for b in res.breaks:
        st = b.statistics or {}; voters = st.get("voting_methods", []); i = b.index
        delta = (1 if b.direction in ("up","increase") else -1) * abs(b.magnitude)
        row = dict(series=key, label=label, col=col, date=pd.Timestamp(b.date).strftime("%Y-%m"), index=i, direction="up" if delta > 0 else "down",
                   k=len(voters), m=len(set(voters)), conf=round(float(b.confidence), 3), delta=round(float(delta), 4),
                   pct=round(float(np.exp(delta) - 1), 4) if col == "log_level" else np.nan, z=round(abs(delta) / sd_step, 2), unit=unit,
                   level_before=round(float(df["level"].iloc[max(0, i-5):i].mean()), 3), level_after=round(float(df["level"].iloc[i:i+5].mean()), 3),
                   voters="|".join(voters), span=np.nan, votes_by_method="")
        if with_replay:
            cl = next((c for c in clusters if min(p[0] for p in c) - 1 <= i <= max(p[0] for p in c) + 1), None)
            if cl:
                row["span"] = max(p[0] for p in cl) - min(p[0] for p in cl)
                row["votes_by_method"] = json.dumps({SHORT[m]: sum(1 for p in cl if p[2] == m) for m in METHODS if any(p[2] == m for p in cl)})
        rows.append(row)
    return pd.DataFrame(rows), res, per_method

main_rows, RES, PER_METHOD = [], {}, {}
for key, (col, label, unit, _) in SERIES.items():
    t0 = dt.datetime.now(); df_k, RES[key], PER_METHOD[key] = analyse(key, col, label, unit); main_rows.append(df_k)
    print(f"{key:12s} {len(df_k)} breaks  ({(dt.datetime.now()-t0).seconds}s):", ", ".join(f"{r.date} {r.direction} k={r.k} m={r.m} c={r.conf:.2f}" for r in df_k.itertuples()))
MAIN = pd.concat(main_rows, ignore_index=True); MAIN.to_csv(os.path.join(OUT, "tables", "breaks_main.csv"), index=False)
MAIN[["series","date","direction","k","m","span","conf","delta","pct","z"]]

cpi_yoy      2 breaks  (0s): 2022-02 up k=18 m=9 c=0.95, 2024-10 down k=7 m=4 c=0.69
policy_rate  1 breaks  (0s): 2023-06 up k=10 m=8 c=0.92
cci          3 breaks  (0s): 2008-06 down k=11 m=6 c=0.67, 2010-04 up k=7 m=6 c=0.69, 2018-09 down k=7 m=7 c=0.67
tur_etf      4 breaks  (0s): 2009-10 up k=6 m=4 c=0.64, 2011-08 down k=5 m=5 c=0.41, 2018-06 down k=7 m=6 c=0.77, 2022-11 up k=6 m=6 c=0.72


,series,date,direction,k,m,span,conf,delta,pct,z
0,cpi_yoy,2022-02,up,18,9,30,0.950,38.4223,NaN,14.23
1,cpi_yoy,2024-10,down,7,4,10,0.686,-17.7917,NaN,6.59
2,policy_rate,2023-06,up,10,8,11,0.922,12.2833,NaN,8.74
3,cci,2008-06,down,11,6,16,0.670,-2.1202,NaN,4.53
4,cci,2010-04,up,7,6,13,0.692,2.3257,NaN,4.96
5,cci,2018-09,down,7,7,7,0.674,-2.9328,NaN,6.26
6,tur_etf,2009-10,up,6,4,12,0.644,0.1651,0.1795,1.66
7,tur_etf,2011-08,down,5,5,3,0.408,-0.2820,-0.2457,2.83
8,tur_etf,2018-06,down,7,6,5,0.769,-0.4568,-0.3667,4.59
9,tur_etf,2022-11,up,6,6,4,0.722,0.4707,0.6011,4.73


## 2. USD/TRY at daily frequency

In [3]:
DAILY = pd.read_csv(os.path.join(DATA, "usdtry_daily.csv"), parse_dates=["Date"])
WINDOWS = {"2021-22": ("2021-01-01", "2022-12-31"), "2023": ("2023-01-01", "2023-12-31")}
drows, DRES = [], {}
for w, (s, e) in WINDOWS.items():
    d = DAILY[(DAILY.Date >= s) & (DAILY.Date <= e)].reset_index(drop=True)
    for col, name in (("log_level", "level"), ("rv21", "realised volatility")):
        dd = d.dropna(subset=[col]).reset_index(drop=True); r = detect(dd, col); DRES[(w, col)] = (dd, r)
        sd_step = np.nanstd(np.diff(dd[col].values))
        for b in r.breaks:
            st = b.statistics or {}; voters = st.get("voting_methods", []); sign = 1 if b.direction in ("up","increase") else -1
            drows.append(dict(window=w, target=name, date=pd.Timestamp(b.date).strftime("%Y-%m-%d"), direction="up" if sign > 0 else "down", k=len(voters), m=len(set(voters)),
                              conf=round(float(b.confidence), 3), delta=round(sign*abs(b.magnitude), 4), pct=round(float(np.exp(sign*abs(b.magnitude))-1), 4) if col == "log_level" else np.nan,
                              z=round(abs(b.magnitude)/sd_step, 2), n_obs=len(dd)))
DAILYB = pd.DataFrame(drows); DAILYB.to_csv(os.path.join(OUT, "tables", "breaks_daily.csv"), index=False); DAILYB

,window,target,date,direction,k,m,conf,delta,pct,z,n_obs
0,2021-22,level,2021-11-22,up,9,6,0.849,0.1259,0.1341,7.50,521
1,2021-22,realised volatility,2021-04-20,down,6,5,0.610,-0.2515,NaN,8.24,521
2,2021-22,realised volatility,2021-11-24,up,7,6,0.700,0.3013,NaN,9.87,521
3,2021-22,realised volatility,2021-12-20,up,7,6,0.690,0.3470,NaN,11.36,521
4,2021-22,realised volatility,2022-01-24,down,9,4,0.721,-0.3395,NaN,11.12,521
5,2023,level,2023-05-12,up,5,5,0.538,0.0084,0.0084,1.13,260
6,2023,level,2023-06-06,up,11,7,0.861,0.0928,0.0972,12.47,260
7,2023,level,2023-10-18,up,5,5,0.586,0.0091,0.0092,1.23,260
8,2023,realised volatility,2023-06-06,up,11,7,0.854,0.1051,NaN,6.56,260
9,2023,realised volatility,2023-06-23,up,5,5,0.526,0.0525,NaN,3.28,260


## 3. Matching to the Table 1 chronology
Rule (Section 3.3): a break matches an episode if its date lies in the episode's period; otherwise if it lies within three months of the period on either side. Periods are the ones documented in Table 1.

In [4]:
EPISODES = [  # name, period start, period end (inclusive, month resolution)
    ("Global financial crisis",        "2008-06", "2010-03"),
    ("Currency crisis 2018",           "2018-05", "2018-09"),
    ("Pandemic shock 2020",            "2020-03", "2020-05"),
    ("Easing and lira collapse 2021",  "2021-09", "2021-12"),
    ("Inflation surge 2022",           "2022-01", "2022-10"),
    ("Return to conventional policy",  "2023-06", "2023-11"),
    ("Disinflation 2024–25",           "2024-06", "2025-12"),
]
def match(date):
    d = pd.Period(date[:7], "M")
    for name, s, e in EPISODES:
        if pd.Period(s, "M") <= d <= pd.Period(e, "M"): return name
    best = None
    for name, s, e in EPISODES:
        gap = min(abs((d - pd.Period(s, "M")).n), abs((d - pd.Period(e, "M")).n))
        if gap <= 3 and (best is None or gap < best[1]): best = (name, gap)
    return best[0] if best else "none"
MAIN["episode"] = MAIN.date.map(match); DAILYB["episode"] = DAILYB.date.map(match)
matrix = pd.DataFrame(index=[e[0] for e in EPISODES], columns=[SERIES[k][1] for k in SERIES] + ["USD/TRY (daily)"]).fillna("")
for r in MAIN.itertuples():
    if r.episode != "none": matrix.loc[r.episode, SERIES[r.series][1]] += (", " if matrix.loc[r.episode, SERIES[r.series][1]] else "") + r.date
for r in DAILYB[DAILYB.target == "level"].itertuples():
    if r.episode != "none": matrix.loc[r.episode, "USD/TRY (daily)"] += (", " if matrix.loc[r.episode, "USD/TRY (daily)"] else "") + r.date
matrix.to_csv(os.path.join(OUT, "tables", "episode_matrix.csv")); MAIN.to_csv(os.path.join(OUT, "tables", "breaks_main.csv"), index=False); DAILYB.to_csv(os.path.join(OUT, "tables", "breaks_daily.csv"), index=False)
print("unmatched monthly breaks:", MAIN[MAIN.episode == "none"][["series","date"]].values.tolist()); matrix

unmatched monthly breaks: [['tur_etf', '2011-08']]


,"CPI inflation (year-on-year, %)",Overnight interbank rate (%),Consumer confidence index,"MSCI Turkey ETF (TUR, USD)",USD/TRY (daily)
Global financial crisis,,,"2008-06, 2010-04",2009-10,
Currency crisis 2018,,,2018-09,2018-06,
Pandemic shock 2020,,,,,
Easing and lira collapse 2021,,,,,2021-11-22
Inflation surge 2022,2022-02,,,2022-11,
Return to conventional policy,,2023-06,,,"2023-05-12, 2023-06-06, 2023-10-18"
Disinflation 2024–25,2024-10,,,,


## 4. LLM attributions and ratings
Live generation needs `OPENROUTER_API_KEY`; otherwise attributions of the 27 May 2026 run are used where the break is identical, and new breaks are marked *pending*.

In [5]:
# Attribution models. PRIMARY is the main model of the paper (Table 3, Section 4.1). The attributions in output/attributions/ were
# generated with BASELINE (Claude Sonnet 4, the model used in the first version of this study) and form the Sonnet 4 column of the
# model comparison (Table 5, Section 4.2); the attributions of PRIMARY and of the other models are in output/attributions/models/<model>/.
KEY = os.environ.get("OPENROUTER_API_KEY")
PRIMARY = "anthropic/claude-opus-5"; BASELINE = "anthropic/claude-sonnet-4"
DESCRIPTIONS = {
 "tur_etf": "iShares MSCI Turkey ETF (ticker TUR), monthly closing price in US DOLLARS. It tracks Turkish equities in USD terms, so it reflects both local market moves and Turkish lira exchange-rate changes.",
 "cpi_yoy": "Turkish consumer price inflation, year-on-year percentage change, monthly. Values are annual inflation rates in percent (e.g. 85 means 85% year-on-year inflation).",
 "policy_rate": "Turkey's overnight interbank / call-money interest rate in percent, monthly; a proxy for the Central Bank of the Republic of Turkey (CBRT) policy stance.",
 "cci": "OECD consumer confidence index for Turkey, monthly, amplitude-adjusted so that about 100 is the long-run average: values above 100 indicate above-average optimism, below 100 pessimism.",
 "usdtry_daily": "USD/TRY exchange rate (Turkish lira per US dollar), daily closing price.",
}
def cached_attribution(series, date):
    if series == "usdtry_daily":
        p = os.path.join(CACHE_DIR, "llm_explanations", "daily_usdtry_level.md")
        return open(p, encoding="utf-8").read() if (date == "2021-11-22" and os.path.exists(p)) else None
    try: ref = pd.read_csv(os.path.join(CACHE_DIR, "detected_breaks.csv"))
    except Exception: return None
    hit = ref[(ref.series == series) & (ref.date.str[:7] == date)]
    if hit.empty: return None
    p = os.path.join(CACHE_DIR, "llm_explanations", f"{series}_break{int(hit.break_no.iloc[0])}.md")
    return open(p, encoding="utf-8").read() if os.path.exists(p) else None
def attribution(series, date, res=None, df=None, col="level"):
    path = os.path.join(OUT, "attributions", f"{series}_{date}.md")
    if os.path.exists(path): return open(path, encoding="utf-8").read(), "file"
    if KEY and os.environ.get("NO_LLM") != "1" and res is not None:
        from structural_break_analyzer.llm.providers.openai_provider import OpenAIProvider
        from structural_break_analyzer.llm.break_explainer import BreakExplainer
        be = BreakExplainer(OpenAIProvider(api_key=KEY, model=BASELINE, base_url="https://openrouter.ai/api/v1"))
        b = next(x for x in res.breaks if pd.Timestamp(x.date).strftime("%Y-%m") == date[:7] or pd.Timestamp(x.date).strftime("%Y-%m-%d") == date)
        i = b.index; before, after = df[col].iloc[max(0, i-5):i].mean(), df[col].iloc[i:i+5].mean()
        b.magnitude, b.direction = abs(after - before), ("up" if after > before else "down"); res.value_col = col
        txt = be.explain_break(b, df, value_col=col, date_col="Date", data_description=DESCRIPTIONS[series]); txt = getattr(txt, "explanation", txt) if not isinstance(txt, str) else txt
        open(path, "w", encoding="utf-8").write(txt); return txt, "live"
    c = cached_attribution(series, date)
    if c: open(path, "w", encoding="utf-8").write(c); return c, "cached"
    return None, "pending"
status = {}
for r in MAIN.itertuples():
    txt, how = attribution(r.series, r.date, RES[r.series], load(r.series)); status[(r.series, r.date)] = how
for r in DAILYB[DAILYB.target == "level"].itertuples():
    dd, res = DRES[(r.window, "log_level")]; txt, how = attribution("usdtry_daily", r.date, res, dd); status[("usdtry_daily", r.date)] = how
print({k: v for k, v in status.items()})
# BASELINE (Sonnet 4) attributions: one-line summary and rating against the record (rubric of Section 3.3)
RATINGS = {
 ("cpi_yoy","2022-02"):     ("Russia's invasion of Ukraine (24 Feb 2022): energy and food prices, lira pressure", "partly accurate"),
 ("cpi_yoy","2024-10"):     ("Delayed transmission of tight 2024 policy; base effects", "accurate"),
 ("policy_rate","2023-06"): ("New economic team; 22 Jun hike from 8.5% to 15%, path toward 45%", "accurate"),
 ("cci","2008-06"):         ("Global financial crisis intensifying; Bear Stearns; oil near $140", "accurate"),
 ("cci","2010-04"):         ("Generic post-crisis recovery narrative, no specific event", "partly accurate"),
 ("cci","2018-09"):         ("August–September 2018 currency crisis: lira fall peaking 13 Aug, inflation surge, September rate increase", "accurate"),
 ("tur_etf","2011-08"):     ("US rating downgrade (5 Aug 2011), euro-area debt crisis; break judged a false positive from 30-month window means", "partly accurate"),
 ("usdtry_daily","2023-05-12"): ("Uncertainty before the elections of 14 May 2023", "accurate"),
 ("usdtry_daily","2023-06-06"): ("Runoff of 28 May 2023; expected continuation of the low-rate policy", "misleading"),
 ("usdtry_daily","2023-10-18"): ("Continued resistance to rate increases; Middle East conflict from 7 Oct; dollar strength", "misleading"),
 ("tur_etf","2009-10"):     ("Post-crisis recovery: near-zero US rates, capital inflows to emerging markets", "accurate"),
 ("tur_etf","2018-06"):     ("2018 currency crisis: June election, Brunson dispute, 10 Aug tariff doubling", "accurate"),
 ("tur_etf","2022-11"):     ("Monetary-policy normalisation 'beginning in late 2022'", "misleading"),
 ("usdtry_daily","2021-11-22"): ("CBRT's 18 Nov cut to 15% amid >20% inflation; capital flight", "accurate"),
}
def rate(series, date):
    s, r = RATINGS.get((series, date), ("pending", "pending")); return s, (r if status.get((series, date)) != "pending" else "pending")
MAIN["baseline_attribution"], MAIN["baseline_rating"] = zip(*[rate(r.series, r.date) for r in MAIN.itertuples()])
DAILYB["baseline_attribution"], DAILYB["baseline_rating"] = zip(*[rate("usdtry_daily", r.date) if r.target == "level" else ("", "") for r in DAILYB.itertuples()])
MAIN.to_csv(os.path.join(OUT, "tables", "breaks_main.csv"), index=False); DAILYB.to_csv(os.path.join(OUT, "tables", "breaks_daily.csv"), index=False)
MAIN[["series","date","episode","baseline_attribution","baseline_rating"]]

{('cpi_yoy', '2022-02'): 'file', ('cpi_yoy', '2024-10'): 'file', ('policy_rate', '2023-06'): 'file', ('cci', '2008-06'): 'file', ('cci', '2010-04'): 'file', ('cci', '2018-09'): 'file', ('tur_etf', '2009-10'): 'file', ('tur_etf', '2011-08'): 'file', ('tur_etf', '2018-06'): 'file', ('tur_etf', '2022-11'): 'file', ('usdtry_daily', '2021-11-22'): 'file', ('usdtry_daily', '2023-05-12'): 'file', ('usdtry_daily', '2023-06-06'): 'file', ('usdtry_daily', '2023-10-18'): 'file'}


,series,date,episode,baseline_attribution,baseline_rating
0,cpi_yoy,2022-02,Inflation surge 2022,Russia's invasion of Ukraine (24 Feb 2022): en...,partly accurate
1,cpi_yoy,2024-10,Disinflation 2024–25,Delayed transmission of tight 2024 policy; bas...,accurate
2,policy_rate,2023-06,Return to conventional policy,New economic team; 22 Jun hike from 8.5% to 15...,accurate
3,cci,2008-06,Global financial crisis,Global financial crisis intensifying; Bear Ste...,accurate
4,cci,2010-04,Global financial crisis,"Generic post-crisis recovery narrative, no spe...",partly accurate
5,cci,2018-09,Currency crisis 2018,August–September 2018 currency crisis: lira fa...,accurate
6,tur_etf,2009-10,Global financial crisis,"Post-crisis recovery: near-zero US rates, capi...",accurate
7,tur_etf,2011-08,none,"US rating downgrade (5 Aug 2011), euro-area de...",partly accurate
8,tur_etf,2018-06,Currency crisis 2018,"2018 currency crisis: June election, Brunson d...",accurate
9,tur_etf,2022-11,Inflation surge 2022,Monetary-policy normalisation 'beginning in la...,misleading


## 4b. Model comparison
The same 14 breaks, prompt and window statistics are sent to further LLMs through OpenRouter (temperature 0.3). Outputs are cached per model under `output/attributions/models/`. Ratings are entered after reading the outputs against the record.

In [6]:
# Ratings of the model-comparison attributions against the record (rubric of Section 3.3): A = accurate, P = partly accurate, M = misleading.
# Read from output/attributions/models/<model>/; keys are (series, date).
_R = {"A": "accurate", "P": "partly accurate", "M": "misleading", "-": ""}
_ORDER = [("cpi_yoy","2022-02"),("cpi_yoy","2024-10"),("policy_rate","2023-06"),("cci","2008-06"),("cci","2010-04"),("cci","2018-09"),
          ("tur_etf","2009-10"),("tur_etf","2011-08"),("tur_etf","2018-06"),("tur_etf","2022-11"),
          ("usdtry_daily","2021-11-22"),("usdtry_daily","2023-05-12"),("usdtry_daily","2023-06-06"),("usdtry_daily","2023-10-18")]
_CODES = {
 "anthropic/claude-opus-5":        "AAAAAAAAAAAAAA",
 "openai/gpt-5.5":                 "AAAAPAAAAAAAAA",
 "google/gemini-3.5-flash-lite":   "AAAAPAAAAAAPAA",
 "openai/gpt-5.4-nano":            "PPPPPPPPPPPPPP",
 "mistralai/mistral-small-2603":   "PMAAMAPAAMAMMM",
 "qwen/qwen3.5-9b":                "MPMAMMMMAM-M-M",   # "-" = no response returned after three attempts
}
EXTRA_MODEL_RATINGS = {m: {k: _R[c] for k, c in zip(_ORDER, codes) if _R[c]} for m, codes in _CODES.items()}
print({m: pd.Series(list(v.values())).value_counts().to_dict() for m, v in EXTRA_MODEL_RATINGS.items()})
# summaries of the PRIMARY (Opus 5) attributions for the "LLM attribution" column of Table 3
OPUS_SUMMARIES = {
 ("cpi_yoy","2022-02"):     "Rate cuts Sep–Dec 2021 into a depreciating lira; pass-through of the Dec 2021 collapse; Jan 2022 administered price and wage increases",
 ("cpi_yoy","2024-10"):     "Base effects from the mid-2023 price surge; lagged effect of tightening to 50%; real appreciation under managed depreciation",
 ("policy_rate","2023-06"): "Runoff 28 May; Şimşek (3 Jun) and Erkan (9 Jun) appointed; 22 Jun hike from 8.5% to 15% and subsequent path",
 ("cci","2008-06"):         "Mid-2008 commodity and inflation shock; CBRT hikes May–Jul 2008; AKP closure case; global financial stress",
 ("cci","2010-04"):         "V-shaped recovery: GDP rebound, falling unemployment, policy rate at 6.5%, equity highs",
 ("cci","2018-09"):         "August 2018 lira crisis (Brunson dispute, 10 Aug tariffs); inflation 24.5% in Sep; 625 bp hike on 13 Sep",
 ("tur_etf","2009-10"):     "Global post-crisis risk-on rally; lira appreciation; CBRT easing to 6.75%; narrowing sovereign spreads",
 ("tur_etf","2011-08"):     "Global risk-off (US downgrade 5 Aug, euro-area crisis); lira depreciation; CBRT cut on 4 Aug",
 ("tur_etf","2018-06"):     "Lira slide from May; emergency hikes 23 May and 7 Jun; election 24 Jun; US–Turkey dispute",
 ("tur_etf","2022-11"):     "Retail equity boom under negative real rates (cuts to 9% by 24 Nov); lira held stable by KKM",
 ("usdtry_daily","2021-11-22"): "18 Nov cut to 15%; 22 Nov presidential defence of low rates; lira −15%; KKM on 20 Dec",
 ("usdtry_daily","2023-05-12"): "FX defence ahead of the 14 May election; post-election pivot and the appointments of 3 and 9 June",
 ("usdtry_daily","2023-06-06"): "End of FX suppression after the runoff; Şimşek 3–4 Jun; 7 Jun 7% fall; Erkan 9 Jun; 22 Jun hike",
 ("usdtry_daily","2023-10-18"): "Controlled depreciation under the new team; hikes to 30% (21 Sep) and 35% (26 Oct); inflation about 61%",
}

{'anthropic/claude-opus-5': {'accurate': 14}, 'openai/gpt-5.5': {'accurate': 13, 'partly accurate': 1}, 'google/gemini-3.5-flash-lite': {'accurate': 12, 'partly accurate': 2}, 'openai/gpt-5.4-nano': {'partly accurate': 14}, 'mistralai/mistral-small-2603': {'misleading': 6, 'accurate': 6, 'partly accurate': 2}, 'qwen/qwen3.5-9b': {'misleading': 9, 'accurate': 2, 'partly accurate': 1}}


In [7]:
# MODEL COMPARISON
MODELS = {"anthropic/claude-opus-5": "Claude Opus 5 (main)", "openai/gpt-5.5": "GPT-5.5", "anthropic/claude-sonnet-4": "Claude Sonnet 4",
          "google/gemini-3.5-flash-lite": "Gemini 3.5 Flash-Lite", "openai/gpt-5.4-nano": "GPT-5.4 nano", "mistralai/mistral-small-2603": "Mistral Small", "qwen/qwen3.5-9b": "Qwen3.5 9B"}
def slug(m): return m.replace("/", "_").replace(".", "-")
def attribution_model(model, series, date, res, df, col="level"):
    folder = os.path.join(OUT, "attributions", "models", slug(model)); os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, f"{series}_{date}.md")
    if os.path.exists(path): return open(path, encoding="utf-8").read(), "file"
    if model == BASELINE:   # baseline = the attributions in output/attributions/
        base = os.path.join(OUT, "attributions", f"{series}_{date}.md")
        if os.path.exists(base): txt = open(base, encoding="utf-8").read(); open(path, "w", encoding="utf-8").write(txt); return txt, "baseline"
    if not KEY or os.environ.get("NO_LLM") == "1": return None, "no key"
    from structural_break_analyzer.llm.providers.openai_provider import OpenAIProvider
    from structural_break_analyzer.llm.break_explainer import BreakExplainer
    be = BreakExplainer(OpenAIProvider(api_key=KEY, model=model, base_url="https://openrouter.ai/api/v1"))
    b = next(x for x in res.breaks if pd.Timestamp(x.date).strftime("%Y-%m") == date[:7] or pd.Timestamp(x.date).strftime("%Y-%m-%d") == date)
    i = b.index; before, after = df[col].iloc[max(0, i-5):i].mean(), df[col].iloc[i:i+5].mean()
    b.magnitude, b.direction = abs(after - before), ("up" if after > before else "down"); res.value_col = col
    try:
        txt = be.explain_break(b, df, value_col=col, date_col="Date", data_description=DESCRIPTIONS[series]); txt = txt if isinstance(txt, str) else getattr(txt, "explanation", str(txt))
    except Exception as e:
        return None, f"error: {str(e)[:80]}"
    open(path, "w", encoding="utf-8").write(txt); return txt, "live"
BREAKS = [(r.series, r.date, RES[r.series], load(r.series)) for r in MAIN.itertuples()] + \
         [("usdtry_daily", r.date, DRES[(r.window, "log_level")][1], DRES[(r.window, "log_level")][0]) for r in DAILYB[DAILYB.target == "level"].itertuples()]
mstatus = {}
for model in MODELS:
    for series, date, res, df in BREAKS:
        _, how = attribution_model(model, series, date, res, df); mstatus[(model, series, date)] = how
print(pd.Series(mstatus).groupby(level=0).value_counts().to_string())
# ratings per model (filled after reading the outputs; baseline ratings from RATINGS)
RATINGS_BY_MODEL = {BASELINE: {k: v[1] for k, v in RATINGS.items()}}
RATINGS_BY_MODEL.update(globals().get("EXTRA_MODEL_RATINGS", {}))
rows = []
for series, date, _, _ in BREAKS:
    key = (series, date); rows.append({"series": series, "date": date, **{MODELS[m]: RATINGS_BY_MODEL.get(m, {}).get(key, "") for m in MODELS}})
T5 = pd.DataFrame(rows); T5.to_csv(os.path.join(OUT, "tables", "table5_ratings_by_model.csv"), index=False)
summary = pd.DataFrame({MODELS[m]: pd.Series(list(RATINGS_BY_MODEL.get(m, {}).values())).value_counts() for m in MODELS}).fillna(0).astype(int)
summary.to_csv(os.path.join(OUT, "tables", "table5_summary.csv")); print(summary)

anthropic/claude-opus-5       file      14
anthropic/claude-sonnet-4     file      14
google/gemini-3.5-flash-lite  file      14
mistralai/mistral-small-2603  file      14
openai/gpt-5.4-nano           file      14
openai/gpt-5.5                file      14
qwen/qwen3.5-9b               file      12
                              no key     2
                 Claude Opus 5 (main)  GPT-5.5  Claude Sonnet 4  \
accurate                           14       13                8   
misleading                          0        0                3   
partly accurate                     0        1                3   

                 Gemini 3.5 Flash-Lite  GPT-5.4 nano  Mistral Small  \
accurate                            12             0              6   
misleading                           0             0              6   
partly accurate                      2            14              2   

                 Qwen3.5 9B  
accurate                  2  
misleading                9  
partly accur

## 5. Tables

In [8]:
def fmt_delta(r):
    if r.unit == "%": return f"{r.pct*100:+.0f}%"
    return f"{r.delta:+.1f} pts" if r.unit == "pts" else f"{r.delta:+.0f} pp"
T3 = pd.DataFrame({"Series": MAIN.label, "Date": pd.to_datetime(MAIN.date).dt.strftime("%b %Y"), "k (m)": [f"{r.k} ({r.m})" for r in MAIN.itertuples()],
                   "Conf.": [f"{c:.0%}" for c in MAIN.conf], "z": MAIN.z.map(lambda v: f"{v:.1f}"), "Δ": [fmt_delta(r) for r in MAIN.itertuples()],
                   "Episode (Table 1)": MAIN.episode, "LLM attribution": [OPUS_SUMMARIES[(r.series, r.date)] for r in MAIN.itertuples()], "Rating": [RATINGS_BY_MODEL[PRIMARY][(r.series, r.date)] for r in MAIN.itertuples()]})
dl = DAILYB[DAILYB.target == "level"]
T3d = pd.DataFrame({"Series": "USD/TRY (daily)", "Date": pd.to_datetime(dl.date).dt.strftime("%d %b %Y"), "k (m)": [f"{r.k} ({r.m})" for r in dl.itertuples()],
                    "Conf.": [f"{c:.0%}" for c in dl.conf], "z": dl.z.map(lambda v: f"{v:.1f}"), "Δ": [f"{p*100:+.0f}%" for p in dl.pct],
                    "Episode (Table 1)": dl.episode, "LLM attribution": [OPUS_SUMMARIES[("usdtry_daily", r.date)] for r in dl.itertuples()], "Rating": [RATINGS_BY_MODEL[PRIMARY][("usdtry_daily", r.date)] for r in dl.itertuples()]})
TABLE3 = pd.concat([T3, T3d], ignore_index=True); TABLE3.to_csv(os.path.join(OUT, "tables", "table3_breaks_attributions.csv"), index=False)
print(TABLE3.to_string(index=False))

                         Series        Date  k (m) Conf.    z        Δ             Episode (Table 1)                                                                                                                        LLM attribution   Rating
CPI inflation (year-on-year, %)    Feb 2022 18 (9)   95% 14.2   +38 pp          Inflation surge 2022 Rate cuts Sep–Dec 2021 into a depreciating lira; pass-through of the Dec 2021 collapse; Jan 2022 administered price and wage increases accurate
CPI inflation (year-on-year, %)    Oct 2024  7 (4)   69%  6.6   -18 pp          Disinflation 2024–25           Base effects from the mid-2023 price surge; lagged effect of tightening to 50%; real appreciation under managed depreciation accurate
   Overnight interbank rate (%)    Jun 2023 10 (8)   92%  8.7   +12 pp Return to conventional policy                            Runoff 28 May; Şimşek (3 Jun) and Erkan (9 Jun) appointed; 22 Jun hike from 8.5% to 15% and subsequent path accurate
      Consumer confi

## 6. Figures

In [9]:
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.edgecolor": "#898781", "axes.linewidth": 0.6})
from matplotlib.lines import Line2D; from matplotlib.patches import Patch
from matplotlib.ticker import ScalarFormatter, NullFormatter, FixedLocator
import matplotlib.dates as mdates
INK, UP, DOWN, SHADE = "#383835", "#d95926", "#2a78d6", "#f0efec"   # validated pair: orange = upward break, blue = downward break
SHORTNAME = {"cpi_yoy": "CPI", "policy_rate": "Rate", "cci": "Confidence", "tur_etf": "ETF", "usdtry": "USD/TRY"}
W = 18 / 2.54
def shade_episodes(ax):
    for name, s, e in EPISODES: ax.axvspan(pd.Timestamp(s), pd.Timestamp(e) + pd.offsets.MonthEnd(0), color=SHADE, zorder=0, lw=0)
def plain_log(ax, ticks):
    ax.set_yscale("log"); ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_major_formatter(ScalarFormatter()); ax.yaxis.set_minor_formatter(NullFormatter())
def label_breaks(ax, items, fmt):
    """items: list of (timestamp, direction, confidence); labels staggered when neighbours are close"""
    items = sorted(items, key=lambda t: t[0]); x0, x1 = ax.get_xlim(); span = x1 - x0; levels = [1.0, 0.87, 0.74]; last_x, lvl = None, -1
    for d, direction, conf in items:
        c = UP if direction in ("up", "increase") else DOWN; ax.axvline(d, color=c, ls="--", lw=1.2, zorder=3)
        x = mdates.date2num(d); lvl = (lvl + 1) % 3 if (last_x is not None and x - last_x < 0.16 * span) else 0; last_x = x
        ax.annotate(fmt(d, conf), (d, levels[lvl]), xycoords=("data", "axes fraction"), xytext=(3, -2), textcoords="offset points", va="top", ha="left",
                    fontsize=7, color=INK, zorder=4, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec=c, lw=0.6))
fmt_m = lambda d, c: f"{d.strftime('%b %Y')}\n{c:.0%}"; fmt_d = lambda d, c: f"{d.strftime('%d %b %Y')} ({c:.0%})"
def items_from(df, brk): return [(pd.Timestamp(df.Date.iloc[r.index]), r.direction, r.conf) for r in brk.itertuples()]

# Figure 1: the four monthly series with consensus breaks and Table 1 episodes
fig, axes = plt.subplots(2, 2, figsize=(W, 0.62*W)); axes = axes.ravel()
for ax, (key, (col, label, unit, letter)) in zip(axes, SERIES.items()):
    df = load(key); ax.set_xlim(pd.Timestamp("2004-01-01"), pd.Timestamp("2026-06-30")); shade_episodes(ax); ax.plot(df.Date, df.level, color=INK, lw=1.1, zorder=2)
    if col == "log_level": plain_log(ax, [15, 20, 30, 40, 50])
    label_breaks(ax, items_from(df, MAIN[MAIN.series == key]), fmt_m); ax.set_title(f"({letter}) {label}", loc="left", fontweight="bold")
    ax.xaxis.set_major_locator(mdates.YearLocator(4)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.legend(handles=[Line2D([0],[0], color=UP, ls="--", lw=1.2, label="upward break"), Line2D([0],[0], color=DOWN, ls="--", lw=1.2, label="downward break"), Patch(fc=SHADE, label="episode in Table 1")],
           loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(os.path.join(OUT, "figures", "fig1_series_breaks.png"), dpi=300, bbox_inches="tight"); fig.savefig(os.path.join(OUT, "figures", "fig1_series_breaks.pdf"), bbox_inches="tight"); plt.close(fig)

# Figure 2: vote composition per consensus break (candidates contributed by each detector)
cols = [f"{SHORTNAME[r.series]}\n{pd.Timestamp(r.date).strftime('%b %Y')}\nk={r.k}, m={r.m}" for r in MAIN.itertuples()]
M = np.zeros((len(METHODS), len(MAIN)), dtype=int)
for j, r in enumerate(MAIN.itertuples()):
    vb = json.loads(r.votes_by_method) if r.votes_by_method else {}
    for i, m in enumerate(METHODS): M[i, j] = vb.get(SHORT[m], 0)
fig, ax = plt.subplots(figsize=(W, 0.42*W))
im = ax.imshow(M, cmap=matplotlib.colors.LinearSegmentedColormap.from_list("blues", ["#ffffff", "#cde2fb", "#86b6ef", "#3987e5", "#184f95"]), vmin=0, vmax=max(4, M.max()), aspect="auto")
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=7); ax.set_yticks(range(len(METHODS))); ax.set_yticklabels([SHORT[m] for m in METHODS]); ax.grid(False)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if M[i, j]: ax.text(j, i, str(M[i, j]), ha="center", va="center", fontsize=7, color="white" if M[i, j] >= 3 else INK)
ax.set_xlabel("consensus break (k candidates from m distinct detectors)"); ax.tick_params(length=0)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "figures", "fig2_votes.png"), dpi=300, bbox_inches="tight"); fig.savefig(os.path.join(OUT, "figures", "fig2_votes.pdf"), bbox_inches="tight"); plt.close(fig)

# Figure 3: daily USD/TRY
fig, axes = plt.subplots(3, 1, figsize=(W, 0.8*W))
panels = [("2021-22", "log_level", "(a) USD/TRY, 2021–2022 (lira per dollar)"), ("2021-22", "rv21", "(b) 21-day realised volatility of USD/TRY, 2021–2022 (% annualised)"), ("2023", "log_level", "(c) USD/TRY, 2023 (lira per dollar)")]
for ax, (w, col, title) in zip(axes, panels):
    dd, res = DRES[(w, col)]; y = dd.level if col == "log_level" else dd.rv21 * 100
    ax.plot(dd.Date, y, color=INK, lw=1.0, zorder=2); ax.set_title(title, loc="left", fontweight="bold"); ax.set_xlim(dd.Date.min(), dd.Date.max())
    label_breaks(ax, [(pd.Timestamp(b.date), b.direction, b.confidence) for b in res.breaks], fmt_d)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3 if w == "2021-22" else 2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.legend(handles=[Line2D([0],[0], color=UP, ls="--", lw=1.2, label="upward break"), Line2D([0],[0], color=DOWN, ls="--", lw=1.2, label="downward break")], loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(os.path.join(OUT, "figures", "fig3_usdtry_daily.png"), dpi=300, bbox_inches="tight"); fig.savefig(os.path.join(OUT, "figures", "fig3_usdtry_daily.pdf"), bbox_inches="tight"); plt.close(fig)
print("figures written:", sorted(os.listdir(os.path.join(OUT, "figures"))))


figures written: ['fig1_series_breaks.pdf', 'fig1_series_breaks.png', 'fig2_votes.pdf', 'fig2_votes.png', 'fig3_usdtry_daily.pdf', 'fig3_usdtry_daily.png']


## 7. Summary

In [10]:
print(f"{len(MAIN)} consensus breaks in the four-series panel; misses: "
      f"{[e for e in matrix.index if (matrix.loc[e] == '').all()]}; unmatched: {MAIN[MAIN.episode=='none'][['series','date']].values.tolist()}")
print("attribution ratings, main model:", pd.Series(list(RATINGS_BY_MODEL[PRIMARY].values())).value_counts().to_dict(), "| baseline (Sonnet 4):", pd.Series(list(RATINGS_BY_MODEL[BASELINE].values())).value_counts().to_dict()); print("daily level breaks:", DAILYB[DAILYB.target=='level'][['window','date','conf']].values.tolist())

10 consensus breaks in the four-series panel; misses: ['Pandemic shock 2020']; unmatched: [['tur_etf', '2011-08']]
attribution ratings, main model: {'accurate': 14} | baseline (Sonnet 4): {'accurate': 8, 'partly accurate': 3, 'misleading': 3}
daily level breaks: [['2021-22', '2021-11-22', 0.849], ['2023', '2023-05-12', 0.538], ['2023', '2023-06-06', 0.861], ['2023', '2023-10-18', 0.586]]
